# LR & XGB for PhysioNet 2012 Challenge

This Google Colab notebook implements LR & XGB for the PhysioNet 2012 Challenge of predicting in-hospital mortality.

In [ ]:
# Import libraries
import os
import torch
import numpy as np
import pandas as pd

from tqdm import tqdm
from torch import nn, optim
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define paths to PhysioNet 2012 data in Google Drive
google_drive_folder = '/content/drive/MyDrive/physionet2012/'
set_a_directory = f"{google_drive_folder}/set-a"
set_b_directory = f"{google_drive_folder}/set-b"
outcomes_a_file = f"{set_a_directory}/Outcomes-a.txt"
outcomes_b_file = f"{set_b_directory}/Outcomes-b.txt"
set_c_directory = f"{google_drive_folder}/set-c"
outcomes_c_file = f"{set_c_directory}/Outcomes-c.txt"

In [ ]:
# Helper functions

def load_outcomes(outcomes_path):
    """Reads the Outcomes file into a pandas DataFrame."""
    df_out = pd.read_csv(outcomes_path)
    df_out.set_index("RecordID", inplace=True)
    return df_out

def time_to_hours(time_str):
    """Convert 'HH:MM' to hours from ICU admission."""
    hh, mm = time_str.split(':')
    return int(hh) + int(mm)/60.0

def load_patient_data(file_path, max_hours=48):
    """
    Load a single patient file.

    - Restricts data to the first 'max_hours' hours.
    - Returns columns: [hour, parameter, value].
    """
    df = pd.read_csv(file_path)
    df['hour'] = df['time'].apply(time_to_hours)
    df = df[df['hour'] <= max_hours].copy()
    df.drop(columns=['time'], inplace=True)
    return df.reset_index(drop=True)

def hourly_resample_and_fill(df_patient, max_hours=48):
    """
    Pivot to shape [0..(max_hours-1) x parameters].
    Forward-fill across missing hours.
    """
    df_patient['hourint'] = df_patient['hour'].astype(int)
    df_patient = df_patient.sort_values('hour', ascending=True)
    df_patient = df_patient.drop_duplicates(subset=['hourint','parameter'], keep='last')

    df_pivot = df_patient.pivot(index='hourint', columns='parameter', values='value')
    df_pivot = df_pivot.reindex(index=range(max_hours), copy=False)
    df_pivot = df_pivot.ffill()

    return df_pivot

def aggregate_hourly_features(df_hourly):
    """
    Produce a single row of aggregated features.

    Returns a series with e.g. param_last, param_mean, etc.
    """
    agg_features = {}

    for col in df_hourly.columns:
        series = df_hourly[col]
        agg_features[f"{col}_last"] = series.iloc[-1]
        agg_features[f"{col}_mean"] = series.mean()
        agg_features[f"{col}_min"]  = series.min()
        agg_features[f"{col}_max"]  = series.max()
        agg_features[f"{col}_count"] = series.count()

    return pd.Series(agg_features, dtype=float)

def build_dataset(raw_folder, outcomes_path, max_hours=48):
    """
    Parse raw data file, produce aggregated features, merge with label.

    Returns a dataframe with one row per patient.
    """
    df_outcomes = load_outcomes(outcomes_path)

    rows = []
    for record_id in tqdm(df_outcomes.index, desc=f"building dataset from {os.path.basename(raw_folder)}"):
        patient_file = os.path.join(raw_folder, f"{record_id}.txt")

        if not os.path.isfile(patient_file):
            row_agg = pd.Series(dtype=float)
        else:
            df_patient = load_patient_data(patient_file, max_hours=max_hours)
            df_hourly = hourly_resample_and_fill(df_patient, max_hours=max_hours)
            row_agg = aggregate_hourly_features(df_hourly)

        in_hosp_death = df_outcomes.loc[record_id, 'in-hospital_death']
        row_agg['inhospitaldeath'] = in_hosp_death
        row_agg['recordid'] = record_id
        rows.append(row_agg)

    df_agg = pd.DataFrame(rows)
    df_agg.set_index('recordid', inplace=True)

    return df_agg

In [ ]:
# Build train & test sets

print("Building training set from set a...")
df_train = build_dataset(set_a_directory, outcomes_a_file, max_hours=48)

print("\nBuilding test/validation set from set b...")
df_test = build_dataset(set_b_directory, outcomes_b_file, max_hours=48)

print("\nBuilding final evaluation set from set c...")
df_final = build_dataset(set_c_directory, outcomes_c_file, max_hours=48)

# Identify columns used as features
feature_cols = [c for c in df_train.columns if c != 'InHospitalDeath']

# Make sure we align the columns in train/test
common_features = sorted(list(set(feature_cols).intersection(df_test.columns)))
X_train = df_train[common_features].fillna(0).values
y_train = df_train['InHospitalDeath'].values

X_test = df_test[common_features].fillna(0).values
y_test = df_test['InHospitalDeath'].values

common_features_final = sorted(list(set(feature_cols).intersection(df_final.columns)))
X_final = df_final[common_features_final].fillna(0).values
y_final = df_final['InHospitalDeath'].values

In [ ]:
# Save/load df_train and df_test (standalone new code chunk)
import pickle

# Define a path to save the pickled dataframes
save_path_train = os.path.join(google_drive_folder, "df_train.pkl")
save_path_test = os.path.join(google_drive_folder, "df_test.pkl")
save_path_final = os.path.join(google_drive_folder, "df_final.pkl")

# Save dataframes
df_train.to_pickle(save_path_train)
df_test.to_pickle(save_path_test)
df_final.to_pickle(save_path_final)

print(f"\nDataframes saved to:\n  {save_path_train}\n  {save_path_test}\n  {save_path_final}")

In [ ]:
# define a path to save the pickled dataframes
save_path_train = os.path.join(google_drive_folder, "df_train.pkl")
save_path_test = os.path.join(google_drive_folder, "df_test.pkl")
save_path_final = os.path.join(google_drive_folder, "df_final.pkl")

# load dataframes
df_train = pd.read_pickle(save_path_train)
df_test = pd.read_pickle(save_path_test)
df_final = pd.read_pickle(save_path_final)


# identify columns used as features
feature_cols = [c for c in df_train.columns if c != 'InHospitalDeath']

# align columns in train/test
common_features = sorted(list(set(feature_cols).intersection(df_test.columns)))
X_train = df_train[common_features].fillna(0).values
y_train = df_train['InHospitalDeath'].values

X_test = df_test[common_features].fillna(0).values
y_test = df_test['InHospitalDeath'].values

common_features_final = sorted(list(set(feature_cols).intersection(df_final.columns)))
X_final = df_final[common_features_final].fillna(0).values
y_final = df_final['InHospitalDeath'].values

In [ ]:
# logistic regression baseline (with threshold sweep)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)
import numpy as np

print("\ntraining logistic regression...")
logreg = LogisticRegression(max_iter=50000, random_state=42, solver='newton-cg')
logreg.fit(X_train, y_train)

# predict probabilities
y_prob = logreg.predict_proba(X_test)[:, 1]

# threshold sweep
print("finding optimal threshold based on f1 score...")
thresholds = np.arange(0.01, 1.0, 0.01)
f1_scores = []

for thresh in thresholds:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_thresh))

# find the best threshold
best_threshold_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_threshold_idx]
best_f1 = f1_scores[best_threshold_idx]
print(f"optimal threshold found: {best_threshold:.2f} (yields f1 = {best_f1:.4f})")

# evaluate using the optimal threshold
y_pred_best_threshold = (y_prob >= best_threshold).astype(int)


Training Logistic Regression (without class weights)...


/usr/local/lib/python3.11/dist-packages/scipy/optimize/_linesearch.py:312: LineSearchWarning: The line search algorithm did not converge
  alpha_star, phi_star, old_fval, derphi_star = scalar_search_wolfe2(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/optimize.py:100: LineSearchWarning: The line search algorithm did not converge
  ret = line_search_wolfe2(


Finding optimal threshold based on F1 score...
Optimal threshold found: 0.25 (yields F1 = 0.5116)


In [ ]:
# evaluate on set c (final)
y_prob_c = logreg.predict_proba(X_final)[:, 1]
y_pred_best_threshold_c = (y_prob_c >= best_threshold).astype(int)

acc_c = accuracy_score(y_final, y_pred_best_threshold_c)
auc_c = roc_auc_score(y_final, y_prob_c)
auprc_c = average_precision_score(y_final, y_prob_c)
precision_c = precision_score(y_final, y_pred_best_threshold_c)
recall_c = recall_score(y_final, y_pred_best_threshold_c)
f1_c = f1_score(y_final, y_pred_best_threshold_c)

print(f"\n[logistic regression w/ optimal threshold ({best_threshold:.2f}) on set c (final)]")
print(f"accuracy   = {acc_c:.4f}")
print(f"auc        = {auc_c:.4f}")
print(f"auprc      = {auprc_c:.4f}")
print(f"precision  = {precision_c:.4f}")
print(f"recall     = {recall_c:.4f}")
print(f"f1         = {f1_c:.4f}")


[Logistic Regression w/ Optimal Threshold (0.25) on set C (FINAL)]
Accuracy   = 0.8253
AUC        = 0.8419
AUPRC      = 0.5048
Precision  = 0.4329
Recall     = 0.6291
F1         = 0.5129


In [ ]:
# xgboost baseline

!pip install xgboost
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)


print("\ntraining xgboost...")
xgb_clf = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
)
xgb_clf.fit(X_train, y_train)


# predict probabilities - needed for threshold sweep
y_prob_xgb = xgb_clf.predict_proba(X_test)[:, 1]

# threshold sweep
print("finding optimal threshold based on f1 score...")
thresholds = np.arange(0.01, 1.0, 0.01)
f1_scores = []

for thresh in thresholds:
    y_pred_thresh = (y_prob_xgb >= thresh).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_thresh))

# find the best threshold
best_threshold_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_threshold_idx]
best_f1 = f1_scores[best_threshold_idx]
print(f"optimal threshold found: {best_threshold:.2f} (yields f1 = {best_f1:.4f})")

# evaluate using the optimal threshold
y_pred_best_threshold = (y_prob_xgb >= best_threshold).astype(int)


Training XGBoost...
Finding optimal threshold based on F1 score...
Optimal threshold found: 0.26 (yields F1 = 0.5342)


In [ ]:
# Evaluate on Set C (Final) using the best threshold
y_prob_xgb_c = xgb_clf.predict_proba(X_final)[:, 1]

# Generate predictions using the best threshold found above
y_pred_xgb_c_best_thresh = (y_prob_xgb_c >= best_threshold).astype(int)

acc_xgb_c = accuracy_score(y_final, y_pred_xgb_c_best_thresh)
auc_xgb_c = roc_auc_score(y_final, y_prob_xgb_c)
auprc_xgb_c = average_precision_score(y_final, y_prob_xgb_c)
precision_xgb_c = precision_score(y_final, y_pred_xgb_c_best_thresh)
recall_xgb_c = recall_score(y_final, y_pred_xgb_c_best_thresh)
f1_xgb_c = f1_score(y_final, y_pred_xgb_c_best_thresh)

print(f"\n[XGBoost w/ Optimal Threshold ({best_threshold:.2f}) on set C (FINAL)]")
print(f"\n[XGBoost on Set C (FINAL)] Accuracy  = {acc_xgb_c:.4f}")
print(f"[XGBoost on Set C (FINAL)] AUC       = {auc_xgb_c:.4f}")
print(f"[XGBoost on Set C (FINAL)] AUPRC     = {auprc_xgb_c:.4f}")
print(f"[XGBoost on Set C (FINAL)] Precision = {precision_xgb_c:.4f}")
print(f"[XGBoost on Set C (FINAL)] Recall    = {recall_xgb_c:.4f}")
print(f"[XGBoost on Set C (FINAL)] F1        = {f1_xgb_c:.4f}")



[XGBoost w/ Optimal Threshold (0.26) on set C (FINAL)]

[XGBoost on Set C (FINAL)] Accuracy  = 0.8522
[XGBoost on Set C (FINAL)] AUC       = 0.8649
[XGBoost on Set C (FINAL)] AUPRC     = 0.5444
[XGBoost on Set C (FINAL)] Precision = 0.4956
[XGBoost on Set C (FINAL)] Recall    = 0.5812
[XGBoost on Set C (FINAL)] F1        = 0.5350


In [ ]:
# =====================================================
# ROC and Calibration Plots for Logistic Regression & XGBoost
# =====================================================
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import roc_auc_score
from sklearn.calibration import calibration_curve

# ----------------------------------------
# ROC Curve: Logistic Regression
# ----------------------------------------
fpr_lr_c, tpr_lr_c, _ = roc_curve(y_final, y_prob_c)
roc_auc_lr_c = auc(fpr_lr_c, tpr_lr_c)

plt.figure(figsize=(6, 6))
sns.lineplot(x=fpr_lr_c, y=tpr_lr_c, color='black', linewidth=2,
             label=f'ROC (AUC = {roc_auc_lr_c:.3f})')
plt.plot([0, 1], [0, 1], 'r--', label='No Skill')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Logistic Regression ROC Curve')
plt.legend(loc='lower right')
plt.show()


In [ ]:
# ----------------------------------------
# ROC Curve: XGBoost
# ----------------------------------------
fpr_xgb_c, tpr_xgb_c, _ = roc_curve(y_final, y_prob_xgb_c)
roc_auc_xgb_c = auc(fpr_xgb_c, tpr_xgb_c)

plt.figure(figsize=(6, 6))
sns.lineplot(x=fpr_xgb_c, y=tpr_xgb_c, color='black', linewidth=2,
             label=f'ROC (AUC = {roc_auc_xgb_c:.3f})')
plt.plot([0, 1], [0, 1], 'r--', label='No Skill')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('XGBoost ROC Curve')
plt.legend(loc='lower right')
plt.show()

In [ ]:

# ----------------------------------------
# Calibration Plot: Logistic Regression
# ----------------------------------------
num_cuts = 24
df_lr_c = pd.DataFrame({"prob": y_prob_c, "label": y_final})
df_lr_c["bin"] = pd.cut(df_lr_c["prob"], bins=24, include_lowest=True)
grouped_lr_c = (
    df_lr_c.groupby("bin")
    .agg(
        observed=("label", "mean"),
        expected=("prob", "mean"),
        n=("label", "count")
    )
    .reset_index()
)
grouped_lr_c["se"] = np.sqrt(grouped_lr_c["observed"] * (1 - grouped_lr_c["observed"]) / grouped_lr_c["n"])

plt.figure(figsize=(6, 6))
plt.errorbar(
    x=grouped_lr_c["expected"],
    y=grouped_lr_c["observed"],
    yerr=1.96 * grouped_lr_c["se"],
    fmt="o",
    capsize=3,
    label="Binned Data"
)
plt.plot([0, 1], [0, 1], "r--", label="Perfect Calibration")
sns.regplot(
    x="expected",
    y="observed",
    data=grouped_lr_c,
    scatter=False,
    lowess=True,
    color="black",
    label="Loess Smooth"
)
plt.xlabel("Expected Probability")
plt.ylabel("Observed Fraction")
plt.title("Logistic Regression Calibration Plot")
plt.legend(loc="best")
plt.show()



In [ ]:

# ----------------------------------------
# Calibration Plot: XGBoost
# ----------------------------------------
df_xgb_c = pd.DataFrame({"prob": y_prob_xgb_c, "label": y_final})
df_xgb_c["bin"] = pd.cut(df_xgb_c["prob"], bins=24, include_lowest=True)
grouped_xgb_c = (
    df_xgb_c.groupby("bin")
    .agg(
        observed=("label", "mean"),
        expected=("prob", "mean"),
        n=("label", "count")
    )
    .reset_index()
)
grouped_xgb_c["se"] = np.sqrt(grouped_xgb_c["observed"] * (1 - grouped_xgb_c["observed"]) / grouped_xgb_c["n"])

plt.figure(figsize=(6, 6))
plt.errorbar(
    x=grouped_xgb_c["expected"],
    y=grouped_xgb_c["observed"],
    yerr=1.96 * grouped_xgb_c["se"],
    fmt="o",
    capsize=3,
    label="Binned Data"
)
plt.plot([0, 1], [0, 1], "r--", label="Perfect Calibration")
sns.regplot(
    x="expected",
    y="observed",
    data=grouped_xgb_c,
    scatter=False,
    lowess=True,
    color="black",
    label="Loess Smooth"
)
plt.xlabel("Expected Probability")
plt.ylabel("Observed Fraction")
plt.title("XGBoost Calibration Plot")
plt.legend(loc="best")
plt.show()
